### Week 6 Reflections

Solution: Taking the 90th percentile of the treatment effect and use it ass a proxy for the Marginal Treatment Effect. 

In order to migitate the optimization bias and the impact of estimated noise, we can use a stasticcal measure like the 90th percentile. Using the 90th percentile of the estimated treatment effects among the untreated, we trim off the extreme outliers that are most likely heavily influenced by random error. This will provide a more stable and realistic upper-bound estimate of what the optimal treatment effect could actually be. This captures the subset of the population that benefits the most without being affected by loud noise points. 

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# 1. Load the dataset (using our prior homework dataset as an example)
df = pd.read_csv('homework_6.1.csv')

# Separate treated (X=1) and untreated (X=0) items
treated = df[df['X'] == 1].reset_index(drop=True)
untreated = df[df['X'] == 0].reset_index(drop=True)

# 2. Fit NearestNeighbors models based on the confounder Z
nn_treated = NearestNeighbors(n_neighbors=1).fit(treated[['Z']])

# 3. Find counterfactuals for the untreated items
# For untreated items: closest treated item corresponds to their counterfactual outcome
_, dist_untreated = nn_treated.kneighbors(untreated[['Z']])
te_untreated = treated.iloc[dist_untreated.flatten()]['Y'].values - untreated['Y']

# 4. Calculate Maximum vs. 90th Percentile Treatment Effect
# The problem: taking the exact max exposes us to extreme noise/bias
naive_max_mte = te_untreated.max()

# The solution: take the 90th percentile as a robust proxy
robust_90th_mte = np.percentile(te_untreated, 90)

print(f"Naive Maximum Treatment Effect  : {naive_max_mte:.4f}")
print(f"Robust 90th Percentile Estimate : {robust_90th_mte:.4f}")

Naive Maximum Treatment Effect  : 2.1725
Robust 90th Percentile Estimate : 1.9280
